# Ch13. Practical Issues
**Forecasting: Principles & Practice (Python Edition)**  
Lab Notebook · [github.com/bcseong2/fpppy-labs](https://github.com/bcseong2/fpppy-labs)

In [1]:
%pip install statsforecast neuralforecast hierarchicalforecast mlforecast utilsforecast


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## [Slide 3] 13.1 Weekly Data: STL + ETS Approach

In [2]:
barrels = pd.read_csv("data/us_gasoline.csv", parse_dates=["ds"])

sf = StatsForecast(
    models=[MSTL(
        season_length=52,
        trend_forecaster=AutoETS(model="ZZN")
    )],
    freq="W",
)
level = [80, 95]
fc = sf.forecast(df=barrels, h=104, level=level)

NameError: name 'pd' is not defined

## [Slide 5] 13.1 Weekly Data: Dynamic Harmonic Regression

In [3]:
from statsforecast.utils import pipeline
from statsforecast.feature_engineering import fourier

barrels_fourier, barrels_futr = pipeline(
    barrels,
    features=[partial(fourier, k=6, season_length=52)],
    freq="W", h=2 * 52,
)
sf = StatsForecast(
    models=[AutoARIMA(seasonal=False)], freq="W"
)
sf = sf.fit(barrels_fourier)
fc_fourier = sf.predict(
    h=2 * 52, X_df=barrels_futr, level=level
)

/Data2/data/jc25/forecast/fpppy-labs/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'pipeline' from 'statsforecast.utils' (/Data2/data/jc25/forecast/fpppy-labs/.venv/lib/python3.11/site-packages/statsforecast/utils.py)

## [Slide 10] 13.2 Croston Variants in Python

In [4]:
j06 = (
    pd.read_csv("data/PBS_unparsed.csv", parse_dates=["Month"])
    .loc[lambda x: x["ATC2"] == "J06"]
    .groupby("Month")
    .agg({"Scripts": "sum"})
    .reset_index()
    .rename(columns={"Month": "ds", "Scripts": "y"})
    .assign(unique_id="J06")
)

sf = StatsForecast(
    models=[
        CrostonClassic(),
        CrostonOptimized(),   # alpha optimized in [0.1, 0.3]
        CrostonSBA(),         # bias-corrected variant
    ],
    freq="MS",
)
fc = sf.forecast(df=j06, h=6)

NameError: name 'pd' is not defined

## [Slide 13] 13.3 Log Transformation for Positive Forecasts

In [5]:
def transform_numeric(df, func, *args, **kwargs):
    numeric = df.select_dtypes(np.number)
    return df.assign(**func(numeric, *args, **kwargs))

egg_prices = pd.read_csv("data/eggs.csv", parse_dates=["ds"])
log_prices = transform_numeric(egg_prices, np.log)

sf = StatsForecast(
    models=[AutoETS(season_length=1, model="ZAZ")],
    freq="Y",
)
level = [80, 95]
fc = (
    sf.forecast(df=log_prices, h=50, level=level)
    .pipe(transform_numeric, np.exp)
)

NameError: name 'pd' is not defined

## [Slide 15] 13.3 Scaled Logit for Bounded Forecasts

In [6]:
def scaled_logit(x, lower=0, upper=1):
    return np.log((x - lower) / (upper - x))

def inv_scaled_logit(x, lower=0, upper=1):
    exp_x = np.exp(x)
    return (upper - lower) * exp_x / (1 + exp_x) + lower

scale = {"lower": 50, "upper": 400}
scaled_prices = transform_numeric(
    egg_prices, scaled_logit, **scale
)
fc = (
    sf.forecast(df=scaled_prices, h=50, level=level)
    .pipe(transform_numeric, inv_scaled_logit, **scale)
)

NameError: name 'egg_prices' is not defined

## [Slide 18] 13.4 Combination: Australian Takeaway Revenue

In [7]:
auscafe = (
    pd.read_csv("data/aus_retail.csv", parse_dates=["Month"])
    .loc[lambda x: x["Industry"].str.contains("Takeaway")]
    .assign(unique_id="auscafe")
    .groupby(["unique_id", "Month"], as_index=False)
    .agg({"Turnover": "sum"})
    .rename(columns={"Turnover": "y", "Month": "ds"})
)
train = auscafe.loc[lambda x: x["ds"] < "2014"]
log_train = transform_numeric(train, np.log)

sf = StatsForecast(
    models=[AutoETS(season_length=12),
            MSTL(season_length=12),
            AutoARIMA(season_length=12)],
    freq="MS",
)
fc = (
    sf.forecast(df=log_train, h=60, level=[80, 95])
    .pipe(transform_numeric, np.exp)
)
model_names = ["AutoETS", "MSTL", "AutoARIMA"]
fc["Combination"] = fc[model_names].mean(axis="columns")

NameError: name 'pd' is not defined

## [Slide 21] 13.4 Conformal Prediction Intervals for Combinations

In [8]:
validation = auscafe.loc[
    lambda x: x["ds"].between("2014", "2015-12")
]
fc_val = (
    sf.forecast(df=log_train, h=24)
    .pipe(transform_numeric, np.exp)
    .assign(Combination=lambda x: x[model_names].mean(axis=1))
    .merge(validation[["unique_id", "ds", "y"]])
)
residuals = fc_val["y"] - fc_val["Combination"]

# Apply quantiles to test forecasts
quantiles = {
    "lo-90": 0.05, "lo-80": 0.1,
    "hi-80": 0.9,  "hi-90": 0.95
}
fc_cp = fc_cp.assign(**{
    f"Combination-{s}":
        fc_cp["Combination"] + residuals.quantile(q)
    for s, q in quantiles.items()
})

NameError: name 'auscafe' is not defined

## [Slide 23] 13.5 Backcasting

In [9]:
t_start, t_end = auscafe["ds"].agg(["min", "max"])
backcasts = auscafe.assign(
    y=auscafe["y"].to_numpy()[::-1]
)
sf = StatsForecast(
    models=[AutoETS(season_length=12)], freq="MS"
)
fc = (
    sf.forecast(df=backcasts, h=15, level=[80, 95])
    .assign(ds=lambda x: t_start - (x["ds"] - t_end))
    .sort_values("ds")
)

NameError: name 'auscafe' is not defined

## [Slide 27] 13.7 Outlier Detection

In [10]:
sf = StatsForecast(
    models=[MSTL(
        season_length=4,
        stl_kwargs={"robust": True}
    )],
    freq="QS",
)
sf = sf.fit(tourism)
dcmp = sf.fitted_[0, 0].model_.assign(ds=tourism["ds"])

NameError: name 'StatsForecast' is not defined

## [Slide 29] 13.7 IQR Rules for Outlier Identification

In [11]:
q1 = dcmp["remainder"].quantile(0.25)
q3 = dcmp["remainder"].quantile(0.75)
iqr = q3 - q1
outliers = dcmp.loc[
    ~dcmp["remainder"].between(q1 - 3*iqr, q3 + 3*iqr)
]

NameError: name 'dcmp' is not defined

In [12]:
tourism_cleaned = tourism.assign(
    y=lambda x: x["y"]
        .where(~x["y"].isin(outliers["data"]))
        .interpolate()
)

NameError: name 'tourism' is not defined